# 02 — Analisis Statistik
**"Terang, Kaya, Sejahtera?"** — menjawab RQ1–RQ4 dengan analisis yang masing-masing menghasilkan visual di dashboard.

| Analisis | Pertanyaan | Output dashboard |
|---|---|---|
| Regresi log-log (OLS, galat baku HC3) | RQ1: seberapa kuat cahaya mencerminkan ekonomi? | Scatter + garis regresi |
| Residual terstandar | RQ1: siapa yang menyimpang? | Peta diverging |
| Kuadran cahaya × kesejahteraan | RQ1–2: terang = sejahtera? | Peta kategori |
| Moran's I & LISA | apakah penyimpangan mengelompok? | Peta LISA |
| PCA + k-means | RQ2: tipologi multivariat | Biplot, parallel coordinates, heatmap |
| Model gravitasi Poisson (PPML) | RQ4: migrasi menuju cahaya? | Kartu koefisien + Sankey |

In [1]:
import json, warnings, numpy as np, pandas as pd, geopandas as gpd
import statsmodels.api as sm, statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from libpysal.weights import Queen, KNN, attach_islands
from esda.moran import Moran, Moran_Local
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
P = ROOT / "data" / "processed"
SEED = 2026
df = pd.read_parquet(P / "kabkota.parquet")
print(df.shape)

(514, 39)


## 1. Sembilan variabel analisis
Transformasi log dipakai pada variabel yang sangat menceng ke kanan (cahaya per kapita, PDRB per kapita, kepadatan). `log1p` dipakai pada cahaya karena 4 kabupaten bernilai 0.

In [2]:
VAR = {
    "ln_cahaya_pk":  ("Cahaya per kapita (ln)", lambda d: np.log1p(d.cahaya_pk)),
    "pct_menyala":   ("% luas menyala", lambda d: d.pct_menyala),
    "pertumbuhan_cahaya": ("Pertumbuhan cahaya 2020–24", lambda d: d.pertumbuhan_cahaya),
    "ln_pdrb_pk":    ("PDRB per kapita (ln)", lambda d: np.log(d.pdrb_kapita_ribu)),
    "laju_pdrb_pct": ("Laju PDRB 2024", lambda d: d.laju_pdrb_pct),
    "ipm":           ("IPM", lambda d: d.ipm),
    "p0":            ("Penduduk miskin (P0)", lambda d: d.p0),
    "ikk":           ("Indeks Kemahalan Konstruksi", lambda d: d.ikk),
    "ln_kepadatan":  ("Kepadatan (ln)", lambda d: np.log(d.kepadatan)),
}
for k, (lab, f) in VAR.items():
    df[k] = f(df)
X = df[list(VAR)]
desc = X.describe().T[["mean", "std", "min", "50%", "max"]].round(3)
desc.index = [VAR[i][0] for i in desc.index]; desc

,mean,std,min,50%,max
Cahaya per kapita (ln),2.075,0.761,0.000,2.155,4.698
% luas menyala,32.309,35.452,0.000,12.277,100.000
Pertumbuhan cahaya 2020–24,41.975,38.141,-47.986,37.029,350.689
PDRB per kapita (ln),10.893,0.709,8.865,10.833,13.819
Laju PDRB 2024,4.661,2.501,-5.670,4.640,30.680
IPM,71.919,6.288,36.300,71.685,88.770
Penduduk miskin (P0),11.169,7.073,2.230,9.320,41.420
Indeks Kemahalan Konstruksi,105.522,29.900,77.030,99.655,379.810
Kepadatan (ln),5.351,1.897,0.000,5.059,9.997


In [3]:
corr = X.corr(method="spearman").round(3)
corr.index = corr.columns = [VAR[i][0] for i in corr.columns]
corr

,Cahaya per kapita (ln),% luas menyala,Pertumbuhan cahaya 2020–24,PDRB per kapita (ln),Laju PDRB 2024,IPM,Penduduk miskin (P0),Indeks Kemahalan Konstruksi,Kepadatan (ln)
Cahaya per kapita (ln),1.000,0.572,-0.187,0.574,0.288,0.547,-0.426,-0.044,0.306
% luas menyala,0.572,1.000,-0.304,0.218,0.336,0.653,-0.420,-0.184,0.926
Pertumbuhan cahaya 2020–24,-0.187,-0.304,1.000,-0.187,-0.092,-0.300,0.121,0.080,-0.319
PDRB per kapita (ln),0.574,0.218,-0.187,1.000,0.157,0.569,-0.544,0.054,0.073
Laju PDRB 2024,0.288,0.336,-0.092,0.157,1.000,0.328,-0.288,0.021,0.302
IPM,0.547,0.653,-0.300,0.569,0.328,1.000,-0.606,-0.073,0.600
Penduduk miskin (P0),-0.426,-0.420,0.121,-0.544,-0.288,-0.606,1.000,0.047,-0.352
Indeks Kemahalan Konstruksi,-0.044,-0.184,0.080,0.054,0.021,-0.073,0.047,1.000,-0.190
Kepadatan (ln),0.306,0.926,-0.319,0.073,0.302,0.600,-0.352,-0.190,1.000


## 2. RQ1 — Regresi log-log cahaya → ekonomi
Model utama: $\ln(\text{PDRBpk}) = \beta_0 + \beta_1 \ln(1+\text{Cahayapk}) + \varepsilon$ dengan galat baku HC3 (tahan heteroskedastisitas).
Model kekokohan: menambah ln(kepadatan) dan IKK.

In [4]:
m1 = smf.ols("ln_pdrb_pk ~ ln_cahaya_pk", data=df).fit(cov_type="HC3")
m2 = smf.ols("ln_pdrb_pk ~ ln_cahaya_pk + ln_kepadatan + ikk", data=df).fit(cov_type="HC3")
print(m1.summary().tables[1]); print("R2 =", round(m1.rsquared, 3))
print(m2.summary().tables[1]); print("R2 =", round(m2.rsquared, 3))
from statsmodels.stats.diagnostic import het_breuschpagan
bp = het_breuschpagan(m1.resid, m1.model.exog); print("Breusch–Pagan p =", round(bp[1], 4))

                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        9.6859      0.074    131.197      0.000       9.541       9.831
ln_cahaya_pk     0.5819      0.035     16.480      0.000       0.513       0.651
R2 = 0.39
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        9.6530      0.185     52.253      0.000       9.291      10.015
ln_cahaya_pk     0.6082      0.039     15.791      0.000       0.533       0.684
ln_kepadatan    -0.0203      0.018     -1.115      0.265      -0.056       0.015
ikk              0.0008      0.001      0.760      0.447      -0.001       0.003
R2 = 0.394
Breusch–Pagan p = 0.0639


### Residual terstandar → status keselarasan
* $z > 1{,}5$: **lebih kaya dari cahayanya** (ekonomi besar tetapi “gelap”, mis. tambang/padat modal)
* $z < -1{,}5$: **lebih terang dari ekonominya**
* lainnya: **selaras**

In [5]:
df["residual"] = m1.resid
df["z_residual"] = (df.residual - df.residual.mean()) / df.residual.std()
df["pdrb_prediksi_ribu"] = np.exp(m1.fittedvalues)
df["status_selaras"] = np.select([df.z_residual > 1.5, df.z_residual < -1.5],
                                 ["Lebih kaya dari cahayanya", "Lebih terang dari ekonominya"], "Selaras")
display(df.status_selaras.value_counts())
display(df.nlargest(8, "z_residual")[["nama", "provinsi", "pdrb_kapita_ribu", "cahaya_pk", "z_residual"]])
display(df.nsmallest(8, "z_residual")[["nama", "provinsi", "pdrb_kapita_ribu", "cahaya_pk", "z_residual"]])

status_selaras
Selaras                         462
Lebih kaya dari cahayanya        32
Lebih terang dari ekonominya     20
Name: count, dtype: int64

,nama,provinsi,pdrb_kapita_ribu,cahaya_pk,z_residual
157,Kota Jakarta Pusat,DKI Jakarta,883715.0,11.380694,4.590054
151,Kabupaten Kepulauan Anambas,Kepulauan Riau,362072.0,3.271330,4.097059
256,Kota Kediri,Jawa Timur,565841.0,9.622563,3.945925
154,Kabupaten Kepulauan Seribu,DKI Jakarta,281266.0,2.886419,3.740251
155,Kota Jakarta Selatan,DKI Jakarta,381187.0,10.543292,3.145232
498,Kabupaten Mimika,Papua Tengah,446333.0,16.513617,2.992075
159,Kota Jakarta Utara,DKI Jakarta,369121.0,13.353772,2.858158
158,Kota Jakarta Barat,DKI Jakarta,253217.0,8.513306,2.609819


,nama,provinsi,pdrb_kapita_ribu,cahaya_pk,z_residual
253,Kabupaten Sampang,Jawa Timur,24484.0,17.700039,-2.319196
254,Kabupaten Pamekasan,Jawa Timur,25424.0,17.400310,-2.234184
252,Kabupaten Bangkalan,Jawa Timur,26066.0,14.965743,-2.040012
311,Kabupaten Sabu Raijua,Nusa Tenggara Timur,18279.0,7.326207,-1.996719
299,Kabupaten Lembata,Nusa Tenggara Timur,15730.0,5.362347,-1.985234
21,Kota Lhokseumawe,Aceh,59122.0,56.095029,-1.900159
478,Kabupaten Pegunungan Arfak,Papua Barat,8662.0,1.015967,-1.854895
483,Kabupaten Tambrauw,Papua Barat Daya,9726.0,1.443642,-1.847851


## 3. Kuadran Cahaya × Kesejahteraan
Indeks kesejahteraan = rata-rata z(IPM) dan −z(P0). Batas kuadran = median nasional (mirip Tipologi Klassen).

In [6]:
z = lambda s: (s - s.mean()) / s.std()
df["indeks_sejahtera"] = (z(df.ipm) - z(df.p0)) / 2
mc, ms = df.ln_cahaya_pk.median(), df.indeks_sejahtera.median()
terang, sejahtera = df.ln_cahaya_pk >= mc, df.indeks_sejahtera >= ms
df["kuadran"] = np.select([terang & sejahtera, ~terang & sejahtera, terang & ~sejahtera],
                          ["Terang – Sejahtera", "Gelap – Sejahtera", "Terang – Kurang sejahtera"], "Gelap – Kurang sejahtera")
display(pd.crosstab(df.pulau, df.kuadran, margins=True))
# sensitivitas: batas kuartil 40/60
lo, hi = df.ln_cahaya_pk.quantile([.4, .6]); print("unit di zona batas 40–60% cahaya:", ((df.ln_cahaya_pk > lo) & (df.ln_cahaya_pk < hi)).sum())

kuadran,Gelap – Kurang sejahtera,Gelap – Sejahtera,Terang – Kurang sejahtera,Terang – Sejahtera,All
pulau,,,,,
Bali-Nusa Tenggara,24,3,5,9,41
Jawa,20,14,24,61,119
Kalimantan,9,9,1,37,56
Maluku,11,4,6,0,21
Papua,32,1,5,4,42
Sulawesi,39,14,12,16,81
Sumatera,44,33,25,52,154
All,179,78,78,179,514


unit di zona batas 40–60% cahaya: 102


## 4. Autokorelasi spasial — Moran's I & LISA (bobot Queen, pulau tanpa tetangga disambung ke 1 tetangga terdekat)

In [7]:
gk = gpd.read_file(P / "kabkota.geojson").merge(df[["kode_bps", "z_residual", "ln_pdrb_pk", "ln_cahaya_pk", "indeks_sejahtera"]], on="kode_bps")
gk = gk.sort_values("kode_bps").reset_index(drop=True)
gm = gk.to_crs(3395)
wq = Queen.from_dataframe(gm, use_index=False, silence_warnings=True)
wk = KNN.from_dataframe(gm.set_geometry(gm.representative_point()), k=1)
w = attach_islands(wq, wk); w.transform = "r"
print("pulau (tanpa tetangga Queen):", len(wq.islands))
moran = {}
for v in ["z_residual", "ln_pdrb_pk", "ln_cahaya_pk", "indeks_sejahtera"]:
    mi = Moran(gk[v], w, permutations=999, seed=SEED) if "seed" in Moran.__init__.__code__.co_varnames else Moran(gk[v], w, permutations=999)
    moran[v] = {"I": round(float(mi.I), 4), "p_sim": round(float(mi.p_sim), 4), "z": round(float(mi.z_sim), 2)}
pd.DataFrame(moran).T

pulau (tanpa tetangga Queen): 32


,I,p_sim,z
z_residual,0.4666,0.001,12.87
ln_pdrb_pk,0.5050,0.001,14.64
ln_cahaya_pk,0.5457,0.001,15.29
indeks_sejahtera,0.6419,0.001,17.95


In [8]:
np.random.seed(SEED)
lisa = Moran_Local(gk["z_residual"], w, permutations=999)
lab = {1: "HH (kaya-dari-cahaya mengelompok)", 2: "LH", 3: "LL (terang-dari-ekonomi mengelompok)", 4: "HL"}
gk["lisa"] = [lab[q] if p < 0.05 else "Tidak signifikan" for q, p in zip(lisa.q, lisa.p_sim)]
gk["lisa_p"] = lisa.p_sim
df = df.merge(gk[["kode_bps", "lisa", "lisa_p"]], on="kode_bps", how="left")
df.lisa.value_counts()

lisa
Tidak signifikan                        409
LL (terang-dari-ekonomi mengelompok)     57
HH (kaya-dari-cahaya mengelompok)        42
LH                                        4
HL                                        2
Name: count, dtype: int64

## 5. RQ2 — PCA dan klasterisasi k-means (9 variabel terstandar)

In [9]:
Z = StandardScaler().fit_transform(X)
pca = PCA(random_state=SEED).fit(Z)
ev = pd.DataFrame({"komponen": [f"PC{i+1}" for i in range(9)], "eigenvalue": pca.explained_variance_.round(3),
                   "varians_pct": (pca.explained_variance_ratio_ * 100).round(2)})
ev["kumulatif_pct"] = ev.varians_pct.cumsum().round(2); display(ev)
k_pc = int((ev.eigenvalue > 1).sum()); print("komponen dengan eigenvalue > 1:", k_pc)
S = pca.transform(Z)
for i in range(3): df[f"PC{i+1}"] = S[:, i]
load = pd.DataFrame(pca.components_[:3].T, index=[VAR[v][0] for v in VAR], columns=["PC1", "PC2", "PC3"]).round(3); load

,komponen,eigenvalue,varians_pct,kumulatif_pct
0,PC1,3.993,44.28,44.28
1,PC2,1.432,15.88,60.16
2,PC3,1.150,12.75,72.91
3,PC4,0.794,8.81,81.72
4,PC5,0.696,7.72,89.44
5,PC6,0.425,4.71,94.15
6,PC7,0.283,3.14,97.29
7,PC8,0.167,1.85,99.14
8,PC9,0.078,0.86,100.00


komponen dengan eigenvalue > 1: 3


,PC1,PC2,PC3
Cahaya per kapita (ln),0.367,-0.336,0.136
% luas menyala,0.357,0.436,0.372
Pertumbuhan cahaya 2020–24,-0.238,-0.164,0.470
PDRB per kapita (ln),0.310,-0.438,-0.005
Laju PDRB 2024,0.074,-0.431,0.589
IPM,0.458,0.020,-0.015
Penduduk miskin (P0),-0.397,0.137,0.203
Indeks Kemahalan Konstruksi,-0.298,0.124,0.403
Kepadatan (ln),0.353,0.507,0.266


In [10]:
res = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=50, random_state=SEED).fit(S[:, :k_pc])
    res.append((k, silhouette_score(S[:, :k_pc], km.labels_), davies_bouldin_score(S[:, :k_pc], km.labels_)))
kres = pd.DataFrame(res, columns=["k", "silhouette", "davies_bouldin"]).round(3); display(kres)
# k dipilih di antara 3–4 (batas palet kategori ramah buta warna) dengan silhouette tertinggi
K = int(kres[kres.k.isin([3, 4])].sort_values("silhouette", ascending=False).k.iloc[0]); print("k terpilih:", K)
km = KMeans(n_clusters=K, n_init=50, random_state=SEED).fit(S[:, :k_pc])
df["klaster_id"] = km.labels_
prof = pd.DataFrame(Z, columns=list(VAR)).groupby(df.klaster_id).mean().round(2)
prof.columns = [VAR[c][0] for c in prof.columns]; prof["n"] = df.klaster_id.value_counts().sort_index(); prof

,k,silhouette,davies_bouldin
0,2,0.374,1.061
1,3,0.422,0.813
2,4,0.446,0.687
3,5,0.368,0.791
4,6,0.378,0.772


k terpilih: 4


,Cahaya per kapita (ln),% luas menyala,Pertumbuhan cahaya 2020–24,PDRB per kapita (ln),Laju PDRB 2024,IPM,Penduduk miskin (P0),Indeks Kemahalan Konstruksi,Kepadatan (ln),n
klaster_id,,,,,,,,,,
0,-2.36,-0.91,2.06,-1.69,-0.17,-3.01,3.10,4.12,-1.28,19
1,0.52,1.19,-0.33,0.25,0.05,0.78,-0.47,-0.16,1.06,186
2,2.27,-0.64,1.81,2.34,5.86,-0.12,0.09,0.23,-1.08,8
3,-0.23,-0.66,0.03,-0.11,-0.18,-0.29,0.09,-0.17,-0.54,301


In [11]:
# Penamaan klaster berdasarkan profil rata-rata (z-score), bukan asumsi — aturan berurutan
names, sisa = {}, list(prof.index)
def ambil(key, asc=False):
    i = prof.loc[sisa, key].sort_values(ascending=asc).index[0]; sisa.remove(i); return i
names[ambil("IPM", asc=True)] = "Pedalaman tertinggal: gelap & mahal"
if K >= 4: names[ambil("Laju PDRB 2024")] = "Kantong pertumbuhan pesat (SDA/IKN)"
names[ambil("% luas menyala")] = "Perkotaan padat & terang"
for i in sisa: names[i] = "Perdesaan berkembang"
df["klaster"] = df.klaster_id.map(names); print(names)
display(df[df.klaster.str.startswith("Kantong")][["nama", "provinsi", "laju_pdrb_pct", "pdrb_kapita_ribu"]])
pd.crosstab(df.klaster, df.pulau)

{np.int32(0): 'Pedalaman tertinggal: gelap & mahal', np.int32(2): 'Kantong pertumbuhan pesat (SDA/IKN)', np.int32(1): 'Perkotaan padat & terang', 3: 'Perdesaan berkembang'}


,nama,provinsi,laju_pdrb_pct,pdrb_kapita_ribu
288,Kabupaten Sumbawa Barat,Nusa Tenggara Barat,12.01,235091.0
358,Kabupaten Kutai Timur,Kalimantan Timur,9.82,344492.0
360,Kabupaten Penajam Paser Utara,Kalimantan Timur,30.68,87144.0
387,Kabupaten Morowali,Sulawesi Tengah,16.26,1003185.0
396,Kabupaten Morowali Utara,Sulawesi Tengah,14.03,236449.0
463,Kabupaten Halmahera Tengah,Maluku Utara,18.40,522766.0
465,Kabupaten Halmahera Selatan,Maluku Utara,23.95,79448.0
475,Kabupaten Teluk Bintuni,Papua Barat,29.22,574984.0


pulau,Bali-Nusa Tenggara,Jawa,Kalimantan,Maluku,Papua,Sulawesi,Sumatera
klaster,,,,,,,
Kantong pertumbuhan pesat (SDA/IKN),1,0,2,2,1,2,0
Pedalaman tertinggal: gelap & mahal,0,0,0,0,19,0,0
Perdesaan berkembang,26,15,45,17,20,62,116
Perkotaan padat & terang,14,104,9,2,2,17,38


## 6. RQ4 — Model gravitasi migrasi (Poisson PPML, 34 provinsi SP2020)

In [12]:
od = pd.read_parquet(P / "migrasi_od.parquet")
pv = pd.read_parquet(P / "provinsi.parquet")
# agregasi ke 34 provinsi lama (pemekaran Papua digabung ke induk)
ke34 = {"Papua Barat Daya": "Papua Barat", "Papua Selatan": "Papua", "Papua Tengah": "Papua", "Papua Pegunungan": "Papua",
        "Kepulauan Bangka Belitung": "Kep. Bangka Belitung"}
d34 = df.assign(p34=df.provinsi.replace(ke34)).groupby("p34").agg(pend=("penduduk_2024", "sum"), rad20=("rad_sum_2020", "sum"),
        pdrb=("pdrb_adhb_miliar", "sum"), ipm=("ipm", "mean"))
d34["cahaya_pk20"] = d34.rad20 / d34.pend * 1000; d34["pdrb_pk"] = d34.pdrb / d34.pend * 1e6
xy = od.drop_duplicates("provinsi_asal").set_index("provinsi_asal")[["lon_asal", "lat_asal"]]
G = od.merge(d34.add_suffix("_o"), left_on="provinsi_asal", right_index=True).merge(d34.add_suffix("_d"), left_on="provinsi_tujuan", right_index=True)
from math import radians, sin, cos, asin, sqrt
def hav(a, b, c, d):
    a, b, c, d = map(radians, [a, b, c, d]); return 2 * 6371 * asin(sqrt(sin((d - b) / 2) ** 2 + cos(b) * cos(d) * sin((c - a) / 2) ** 2))
G["jarak_km"] = [hav(r.lon_asal, r.lat_asal, r.lon_tujuan, r.lat_tujuan) for r in G.itertuples()]
f = "jumlah_migran ~ np.log(pend_o) + np.log(pend_d) + np.log(jarak_km) + np.log(cahaya_pk20_o) + np.log(cahaya_pk20_d)"
grav = smf.glm(f, data=G, family=sm.families.Poisson()).fit(cov_type="HC1")
print(grav.summary().tables[1]); print("pasangan OD:", len(G))

                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                -3.2505      1.093     -2.973      0.003      -5.394      -1.107
np.log(pend_o)            0.5499      0.042     12.978      0.000       0.467       0.633
np.log(pend_d)            0.6478      0.062     10.391      0.000       0.526       0.770
np.log(jarak_km)         -1.2090      0.080    -15.164      0.000      -1.365      -1.053
np.log(cahaya_pk20_o)     0.3805      0.124      3.069      0.002       0.138       0.624
np.log(cahaya_pk20_d)    -0.1139      0.138     -0.824      0.410      -0.385       0.157
pasangan OD: 1091


## 7. Simpan hasil analisis untuk dashboard

In [13]:
ring = {
    "n": int(len(df)),
    "ols": {"beta": round(float(m1.params["ln_cahaya_pk"]), 4), "se": round(float(m1.bse["ln_cahaya_pk"]), 4),
            "p": float(m1.pvalues["ln_cahaya_pk"]), "r2": round(float(m1.rsquared), 4), "const": round(float(m1.params["Intercept"]), 4),
            "bp_p": round(float(bp[1]), 4)},
    "ols_robust": {k: round(float(v), 4) for k, v in m2.params.items()} | {"r2": round(float(m2.rsquared), 4)},
    "moran": moran,
    "status": df.status_selaras.value_counts().to_dict(),
    "kuadran": df.kuadran.value_counts().to_dict(),
    "lisa": df.lisa.value_counts().to_dict(),
    "pca": {"varians_pct": ev.varians_pct.tolist(), "eigen": ev.eigenvalue.tolist(), "k_pc": k_pc,
            "loadings": {VAR[v][0]: [round(float(x), 4) for x in pca.components_[:2, i]] for i, v in enumerate(VAR)}},
    "kmeans": {"k": K, "tabel": kres.to_dict("records"), "nama": {int(k): v for k, v in names.items()}},
    "gravitasi": {k: {"coef": round(float(grav.params[k]), 4), "p": float(grav.pvalues[k])} for k in grav.params.index},
    "var_label": {k: v[0] for k, v in VAR.items()},
}
(P / "ringkasan_analisis.json").write_text(json.dumps(ring, ensure_ascii=False, indent=2), encoding="utf-8")
keep = [c for c in df.columns if c not in ("geometry",)]
df[keep].to_parquet(P / "kabkota_analisis.parquet", index=False); df[keep].to_csv(P / "kabkota_analisis.csv", index=False)
prof.to_csv(P / "profil_klaster.csv")
print("tersimpan:", sorted(p.name for p in P.iterdir()))

tersimpan: ['kabkota.csv', 'kabkota.geojson', 'kabkota.parquet', 'kabkota_analisis.csv', 'kabkota_analisis.parquet', 'migrasi_matriks.parquet', 'migrasi_neto.parquet', 'migrasi_od.parquet', 'profil_klaster.csv', 'provinsi.csv', 'provinsi.geojson', 'provinsi.parquet', 'ringkasan_analisis.json', 'sektor_provinsi.parquet']
